# Vector DB

In this notebook, we use a containerized version of Chroma DB. To set up, you will need the following:

1. Install [Docker Desktop](https://www.docker.com/products/docker-desktop/) by following the link and Download Docker Desktop for your operating system.
2. In VS Code's terminal window, navigate to the folder ./05_src/deploying_ai_data. For example, on Mac, you would use `cd ./05_src/deploying_ai_data`.
3. Run the command `docker compose up -d`. This command: 
+ if it doesn't already exist, downloads a Docker image (blueprint) from the internet, specifically from DockerHub (like GitHub but for Docker containers) 
+ inside the container, initiates a number of servers using the specs from ./05_src/deploying_ai_data/docker-compose.yml (a Chroma DB server, a postgres database (postgres is a database management system), pgadmin (to interact with postgres) etc). Have a look inside the file to see what servers and specs it contains. 
+ Note that Docker containers should be thought of as temporary instead of permanent. 

Some of the code chunks in this notebook are not meant to be run: they are contained within `if use_api = True` if statements, but `use_api` has been explicitly set to `False`. This is as this prompt is extremely expensive token-wise. Instead we will import the already-created openAI ouptut ./05_src/deploying_ai_data/documents/pitchfork/chroma_inputs.jsonl later on in the notebook instead.

## Downloading Batch Results

In the previous notebook, we had created batch processes (we split pitchfork reviews into chunks using recursive character splitting). We will start by consulting the status of our batch processes by identifying them throught their descriptions.

In [2]:
%load_ext dotenv
%dotenv ../../05_src/.secrets
%dotenv ../../05_src/.env

import sys
sys.path.append('../../05_src/')

In [22]:
from utils.clients import get_client
from IPython.display import display, Markdown
from tqdm import tqdm
import json
import os

client = get_client(use_gateway=True)
MODEL = os.getenv('MODEL')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')
DOC_DIR = "../../05_src/documents/pitchfork/"
use_api = False

the below code chunk processes some of the outputs from processing the batch, and turns it into the form of a list and saves it as batch_info

but it only does so for batches where batch_description matches the batch_description tag that was created in notebook 04_6 (with a unique id sequence and timestamp)

In [ ]:
if use_api:
    batch_description = 'Pitchfork reviews content embeddings (deploying-ai-8192e796b86242b084df7200276ef5f2) 2026-06-13 23:11:19'
    batch_processes = client.batches.list().to_dict() #note it's using the batches api (client.batches) rather than the files api (client.files)
    batch_info= [
        {'batch_id': batch['id'],
        'description': batch['metadata']['description'],
        'status': batch['status'],
        'request_counts': batch['request_counts'],
        'output_file_id': batch['output_file_id'],
        'input_file_id': batch['input_file_id']}
                for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
        ]
    batch_info

I don't get the output since use_api is False, but batch_info contains the specified batch_info (batch_id, description, status etc) for each batch

When the status of the batches is complete, we can query the `output_file_id` where their results will be stored.

More generally, we will require the original text and the embeddings of that original text mapped through the `custom_id`.

In [26]:
if use_api:
    batch_complete = [
        batch  for batch in batch_info if batch['status'] == 'completed'
    ]
    batch_incomplete = [
        batch  for batch in batch_info if batch['status'] != 'completed'
    ]
    print(f"Batch complete: {len(batch_complete)}\n\nBatch incomplete: {len(batch_incomplete)}")

The output above would've told you that Batch complete = 20 and Batch incomplete = 0 -- just a way to check progress / that there are no errors 

Before we download all results, examine the response of the file API:

In [ ]:
if use_api:
    response = client.files.content(batch_complete[0]['output_file_id']) #note: switching back to files api as we're trying to look at a single file rather than the batches
    text_response = response.text
    lines = text_response.split('\n')
    json.loads(lines[0])

In [ ]:
json.loads(lines[0])

this code chunk doesn't work either, but the output would have various IDs, a response code (200, =success), openAI model used, # of tokens used, and crucially, the embedding of the given file 

The embedding is the intermediate goal of all of this - you've converted the text of a section of the review into its 'essence', and in this vectorized form you can do similarity searches etc that a user may request.

Note it's very important to keep track of the number of tokens you're using, so track and optimize spending. e.g. you will find that some models are cheaper than others (e.g. text-embedding-3-small, used here, is very cheap but sufficient for this purpose) and certain implementations of your code will save tokens (e.g. processing in batches is cheaper)

For our results database, we will need to map the original text to their embeddings. This mapping can be done thanks to a `custom_id` that uniquely links pairs of texts and embeddings. 

In [8]:
def get_text_and_embeddings(batch):
    embedding_lines =  get_content_from_file(batch, 'output_file_id')
    text_lines = get_content_from_file(batch, 'input_file_id')
    return embedding_lines, text_lines

def get_content_from_file(batch, key):
    file = client.files.content(batch[key])
    text = file.text
    lines = text.split('\n')
    content_lines = [json.loads(line) for line in lines if line.strip()]
    return content_lines


Notice that the response is also a .jsonl file. Therefore, we can process it line-by-line and use the `custom_id` to map to the original document chunk.

The function below:

- Creates a dictionary, `text_dict`, with keys given by each `custom_id` and value equal to the text.
- Iterate over all embedding items and use the dictionary defined above to map the embeddings to their input text.

Note how short each of these functions are. This is good coding - short functions with a small footprint (=few inputs) but long, descriptive names means simple and readable code. 

- LLMs tend to create long, complex functions. Part of Jesus' job is actually to break these functions down into simpler ones! 

In [ ]:
#this function extracts reviewid from custom_id (remember that custom_id is reviewid + seq number + start index)
def get_reviewid_from_custom_id(custom_id:str):
    return custom_id.split('_')[0]

def create_chroma_inputs(embedding_lines, text_lines, metadata_dict):
    chroma_inputs = []
    text_dict = {item['custom_id']: item['body']['input'] for item in text_lines}
    for embed_item in embedding_lines:
        custom_id = embed_item['custom_id']
        text = text_dict.get(custom_id, "")
        reviewid = get_reviewid_from_custom_id(custom_id)
        chroma_input = {
            'id': custom_id,
            'embedding': embed_item['response']['body']['data'][0]['embedding'],
            'text': text,
            'metadata': metadata_dict.get(reviewid, {})
        }
        chroma_inputs.append(chroma_input)
    return chroma_inputs

A couple of functions to control the logic flow (=sequence of steps that's run one after the other):

In [10]:
def process_batch_for_chromadb(batch, metadata_dict):
    embedding_lines, text_lines = get_text_and_embeddings(batch)
    chroma_inputs = create_chroma_inputs(embedding_lines, text_lines, metadata_dict)
    return chroma_inputs

def process_batches_for_chromadb(batches, metadata_dict):
    all_chroma_inputs = []
    for batch in tqdm(batches, desc="Processing batches"):
        chroma_inputs = process_batch_for_chromadb(batch, metadata_dict)
        all_chroma_inputs.extend(chroma_inputs)
    return all_chroma_inputs

def chroma_inputs_to_jsonl(chroma_inputs, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in chroma_inputs:
            json_line = json.dumps(item)
            f.write(json_line + '\n')

# Metadata

We enrich each ChromaDB entry with structured metadata joined from the pitchfork source files:

| Field | Source file | Description |
|-------|------------|-------------|
| `reviewid` | pitchfork_reviews | Unique review identifier |
| `album` | pitchfork_reviews | Album title |
| `artist` | pitchfork_reviews | Artist name |
| `score` | pitchfork_reviews | Pitchfork score (0–10) |
| `genre` | pitchfork_genres | Genre(s), comma-separated |
| `label` | pitchfork_labels | Record label |
| `year` | pitchfork_years | Album release year |

Storing metadata directly in ChromaDB allows native filtering with `where` clauses — no additional database query is needed at retrieval time.
+ remember `where` is a sql function! 

In [ ]:
#This function stitches together the text chunk with its associated metadata using a unique reviewid. 
#If we didn't happen to have a reviewid ready-made, we can use uuid or a hash of the data (a unique identifier created deterministically from the contents of the entry)
def load_metadata(doc_dir:str) -> dict:
    """Join pitchfork_*.jsonl files into a metadata dict keyed by str(reviewid)."""
    reviews = {str(r['reviewid']): r for r in load_jsonl(os.path.join(doc_dir, 'pitchfork_reviews.jsonl'))}
    genres_raw = load_jsonl(os.path.join(doc_dir, 'pitchfork_genres.jsonl'))
    labels_raw = load_jsonl(os.path.join(doc_dir, 'pitchfork_labels.jsonl'))
    years_raw  = load_jsonl(os.path.join(doc_dir, 'pitchfork_years.jsonl'))

    genres = {}
    for g in genres_raw:
        rid = str(g['reviewid'])
        genres.setdefault(rid, []).append(g.get('genre') or '')

    labels = {}
    for l in labels_raw:
        rid = str(l['reviewid'])
        if rid not in labels:
            labels[rid] = l.get('label') or ''

    years = {str(y['reviewid']): y.get('year') or 0 for y in years_raw}

    metadata = {}
    for rid, r in reviews.items():
        metadata[rid] = {
            'reviewid': rid,
            'album':    str(r.get('title')  or ''),
            'artist':   str(r.get('artist') or ''),
            'score':    float(r.get('score') or 0.0),
            'genre':    ', '.join(g for g in genres.get(rid, []) if g),
            'label':    str(labels.get(rid) or ''),
            'year':     int(years.get(rid) or 0),
        }
    return metadata

def load_jsonl(file:str):
    data = []
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

Make sure `chroma_inputs.jsonl` has been placed in `05_src/documents/pitchfork/` (provided by the instructional team). This file contains pre-computed embeddings and metadata for all reviews, so students can load the collection without making any API calls.

Set `use_api = True` to regenerate it from the batch API results instead. Note that batch jobs can take up to 24 hours to complete.

Now, we can create our input dictionaries. Since we can't use openAI for this (or if we already have the output and have it saved locally), we just load the chraom_inputs.jsonl file from our local directory. 

In [12]:
out_path = os.path.join(DOC_DIR, "chroma_inputs.jsonl")

if use_api:
    os.makedirs(DOC_DIR, exist_ok=True)
    metadata_dict = load_metadata(DOC_DIR)
    chroma_inputs = process_batches_for_chromadb(batch_complete, metadata_dict)
    chroma_inputs_to_jsonl(chroma_inputs, out_path)
else:
    metadata_dict = load_metadata(DOC_DIR)
    with open(out_path, "r", encoding="utf-8") as f:
        chroma_inputs = [json.loads(line) for line in f]
    for item in chroma_inputs:
        if not item.get("metadata"):
            reviewid = get_reviewid_from_custom_id(item["id"])
            item["metadata"] = metadata_dict.get(reviewid, {})

In [13]:
chroma_inputs[1]

{'id': '18904_3503_0',
 'embedding': [0.034942626953125,
  0.029693603515625,
  -0.0185546875,
  0.03692626953125,
  0.02685546875,
  -0.004047393798828125,
  0.049560546875,
  0.035888671875,
  -0.04132080078125,
  0.0249786376953125,
  0.0238494873046875,
  -0.037872314453125,
  -0.01580810546875,
  -0.003936767578125,
  -0.0296783447265625,
  0.053619384765625,
  -0.0121917724609375,
  -0.06939697265625,
  -0.0169677734375,
  0.0173187255859375,
  0.028778076171875,
  -0.015625,
  -0.01361083984375,
  0.05157470703125,
  0.00914764404296875,
  0.0264434814453125,
  -0.046905517578125,
  0.057830810546875,
  0.0209503173828125,
  -0.031585693359375,
  -0.0143280029296875,
  -0.0206146240234375,
  0.005786895751953125,
  -0.002166748046875,
  -0.0187530517578125,
  0.0196990966796875,
  0.0548095703125,
  -0.03216552734375,
  -0.039398193359375,
  0.00954437255859375,
  -0.024169921875,
  0.00574493408203125,
  0.066650390625,
  0.041961669921875,
  0.045501708984375,
  -0.00269317626

If you expand this output in a text editor, you'll see the original text chunk, the embedding, and all the metadata info.

# Load Embeddings to Chroma

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os

COLLECTION_NAME = "pitchfork_reviews"
CHROMA_URL = os.getenv("CHROMA_URL") #note that CHROMA_URL is set to port 8000 of your computer in 05_src/.env, i.e. you're connecting to the chromaDB locally (in the Docker container you created). 

def setup_collection(chroma_url:str=CHROMA_URL,
                     collection_name: str = COLLECTION_NAME):
    chroma_client = chromadb.HttpClient(host=chroma_url)
    collections = chroma_client.list_collections()
    if collection_name in [col.name for col in collections]: #if collection already exists, delete it. you could instead write the function to break if the collection already exists, for example.
        chroma_client.delete_collection(name=collection_name)

    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=OpenAIEmbeddingFunction(
            api_key = os.getenv("OPENAI_API_KEY"),
            model_name=EMBEDDING_MODEL)
        )
    return collection

def load_embeddings_to_db(chroma_inputs:list[dict],
                          collection_name:str,
                          chroma_url:str=CHROMA_URL,
                          batch_size:int=1000
                          ):
    collection = setup_collection(chroma_url=chroma_url, collection_name=collection_name)

    for i in tqdm(range(0, len(chroma_inputs), batch_size)):
        batch = chroma_inputs[i:i + batch_size]
        collection.add(
            documents=[item['text'] for item in batch],
            embeddings=[item['embedding'] for item in batch],
            metadatas=[item['metadata'] for item in batch],
            ids=[item['id'] for item in batch]
        )


In [16]:
load_embeddings_to_db(chroma_inputs=chroma_inputs,
                      collection_name=COLLECTION_NAME,
                      chroma_url=CHROMA_URL, 
                      batch_size=1000)

100%|██████████| 20/20 [00:28<00:00,  1.41s/it]


# Prompt Generator

Here we create a prompt with the context gathered through the different data operations.

This part is the next step of our tool: how does a user query our database? Here we are implementing a simpler approach (vector search). notebook 04_8 will use a better and more complex approach so look out for that 
+ Vector search - queries are converted into embeddings, which will be compared to the embeddings in the database; the most similar results (cosine distance closest to 0) will be returned.

In [17]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

USE_GATEWAY = os.getenv("USE_GATEWAY", "False").lower() == "true"

chroma = chromadb.HttpClient(host=CHROMA_URL)
if USE_GATEWAY:
    embedding_function = OpenAIEmbeddingFunction(
        api_key="any value",
        api_base="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_type="openai",
        model_name=EMBEDDING_MODEL,
        default_headers={
            "x-api-key": os.getenv("API_GATEWAY_KEY")
        }
    )
else:
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name=EMBEDDING_MODEL
    )

collection = chroma.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)

In [18]:
collection.query(
    query_texts=["A great album with stunning vocals and production."],
    n_results=3
)

{'ids': [['5502_17666_2295', '185_18098_4602', '398_15632_1550']],
 'distances': [[0.47434413, 0.50253934, 0.51130044]],
 'embeddings': None,
 'metadatas': [[{'label': 'thule iceland',
    'album': 'yesterday was dramatic - today is ok',
    'year': 2001,
    'genre': 'rock, electronic',
    'score': 9.1,
    'reviewid': '5502',
    'artist': 'mm'},
   {'label': 'melankolic',
    'genre': 'global',
    'album': 'living in the flood',
    'artist': 'horace andy',
    'year': 2000,
    'score': 7.0,
    'reviewid': '185'},
   {'artist': 'azure ray',
    'reviewid': '398',
    'label': 'saddle creek',
    'genre': 'rock',
    'album': 'the drinks we drank last night ep',
    'year': 2003,
    'score': 6.0}]],
 'documents': [['an amazing album, what really seals it for me is the     melody. This album is simply bursting with gorgeous ones, penetrating the     dense background and making it one of the most deeply, purely emotionally     affecting albums of the year.  Listening to this album

Next, create prompts that supply the model with more context and dev instructions 

In [19]:
def get_context_data(query:str, collection:chromadb.api.models.Collection, top_n:int):
    results = collection.query(
        query_texts=[query],
        n_results=top_n
    )
    context_data = []
    for idx in range(len(results['ids'][0])):
        details = dict(results['metadatas'][0][idx])
        details['text'] = results['documents'][0][idx]
        context_data.append(details)
    return context_data

def generate_prompt(query:str, collection:chromadb.api.models.Collection, top_n:int):
    context_data = get_context_data(query, collection, top_n)
    prompt = f"Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to {top_n} album reviews.\n\n"
    prompt += f"The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.\n\n"
    prompt += f"<query>{query}</query>\n\n"
    prompt += "<context>\n"
    for k, context in enumerate(context_data):
        prompt += f"<album {k}>\n"
        prompt += f"- Album Title: {context.get('album', 'N/A')}\n"
        prompt += f"- Album Artist: {context.get('artist', 'N/A')}\n"
        prompt += f"- Album Score: {context.get('score', 'N/A')}\n"
        prompt += f"- Genre: {context.get('genre', 'N/A')}\n"
        prompt += f"- Label: {context.get('label', 'N/A')}\n"
        prompt += f"- Release Year: {context.get('year', 'N/A')}\n"
        prompt += f"- Review Quote: {context.get('text', 'N/A')}\n"
        prompt += f"</album {k}>\n\n"
    prompt += "</context>\n\n"
    prompt += "\nBased on the context and nothing else, provide a detailed response to the query."
    return prompt

def generate_response(query:str, collection:chromadb.api.models.Collection, top_n:int=1):
    prompt = generate_prompt(query, collection, top_n)
    print("Generated Prompt:\n", prompt)
    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful assistant that provides information based on Pitchfork reviews.",
        input=[{"role": "user", "content": prompt}],
        max_output_tokens=500,
        temperature=0.7
    )
    return response.output_text


# Query

We can now use chroma's similarity function to query the database. Notice that the query itself needs to be converted to embeddings, so we must provide an `embedding_function`. In this case, we use `OpenAIEmbeddingFunction()` to get compatible embeddings using model `text-embedding-3-small`.

In [23]:
response = generate_response("What are some highly rated albums by emerging indie artists?", 
                             collection, 
                             3)

Generated Prompt:
 Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to 3 album reviews.

The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.

<query>What are some highly rated albums by emerging indie artists?</query>

<context>
<album 0>
- Album Title: conductor
- Album Artist: the comas
- Album Score: 8.0
- Genre: rock
- Label: yep roc
- Release Year: 2004
- Review Quote: At the heart of the indie mindset is a profound disconnect between perception and actuality. The ideal:
    D.I.Y. ethics, an emphasis on art above marketability, and a purity of purpose that is resistant to fads and
    monetary influence. The reality: indie bands are receiving six-digit recording advances from major media
    conglomerates, and are just as ripe for demographic targeting, s

In [24]:
display(Markdown(response))

Here are some highly rated albums by emerging indie artists based on Pitchfork reviews:

1. **Espers - *Espers*** (Score: 8.4)
   Released in 2004 under the Locust label, this album represents a significant entry in the neo-folk revival movement. The review highlights the band's ability to blend psych-charged melancholia with beautifully baroque melodies, led by Greg Weeks' songwriting and the enchanting vocals of Meg Baird. The incorporation of varied instruments like finger cymbals and harmonica provides substantial depth, making it an essential listen for fans of the indie folk genre.

2. **The Comas - *Conductor*** (Score: 8.0)
   Also released in 2004 by Yep Roc, *Conductor* explores the intersection of indie and mainstream music. The review discusses how the band navigates the complexities of the indie music landscape, producing dreamy and rocking pop tunes that appeal to a broad audience. The album's themes, influenced by personal experiences and relationships, resonate well within the indie ethos, making it a noteworthy recommendation.

3. **The Delgados - *Hate*** (Score: 8.1)
   Released in 2003 on the Mantra label, *Hate* showcases the band's unique Scottish melodies and orchestral pop sensibilities. The review notes that while the production might fit into a more "indie adult-contemporary" genre, The Delgados maintain a distinct sound characterized by anthemic and resigned melodies. Their ability to combine grand orchestration with emotional songwriting sets this album apart, making it a must-listen for those interested in the evolution of indie rock.

These albums not only received high scores but also reflect the diverse sounds and themes emerging from the indie scene in the early 2000s. Each album offers a unique listening experience that captures the essence of indie music's exploration of personal and societal themes.

**Note**: Try changing the top_n parameter to 1 and re-run the query. 

In summary, what we have done in this notebook is a RAG: we retrieve data from an external database, pitchfork (that's not whatever openAI is trained on), and use this to generate a response.